## Check Gen-ai model

In [11]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

api_key = os.getenv("GENAI_API_KEY")

# Check whether the api_key got value
if not api_key:
    print("[INFO] API key not found in environment variables")
    raise
else:
    print("[INFO] API key is successfully loaded!")

# Initialize the ChatOpenAI model with OpenRouter's details
llm_model = ChatOpenAI(
    model="openrouter/free",  # The specific model name from OpenRouter
    base_url="https://openrouter.ai/api/v1", # OpenRouter's API endpoint
    api_key=api_key
)
llm_model2 = ChatOpenAI(
    model="cohere/north-mini-code:free",  # The specific model name from OpenRouter
    base_url="https://openrouter.ai/api/v1", # OpenRouter's API endpoint
    api_key=api_key,
    temperature=0.2
)

messages = [
    (
        "system",
        "Your PostgreSQL expert, you can write sql code correctly as per user's query",
    ),
    ("human", "write sql command to get current system timestamp"),
]

response = llm_model2.invoke(messages)

print(response)

[INFO] API key is successfully loaded!
content="Here are a few ways to retrieve the current system timestamp in PostgreSQL. Choose the one that best fits what you need:\n\n### 1. Current date‑time with time zone (most common)\n```sql\nSELECT CURRENT_TIMESTAMP;\n-- or the same, using the alias:\nSELECT NOW() AS current_timestamp;\n```\n- Returns something like `2023‑09‑15 14:32:05.123456+00`\n\n### 2. Current date‑time **without** time zone\n```sql\nSELECT CURRENT_DATE;      -- just the date (YYYY‑MM‑DD)\nSELECT CURRENT_TIME;      -- just the time (HH:MM:SS)\nSELECT LOCALTIMESTAMP;   -- timestamp without time zone, based on the session’s timezone\n```\n\n### 3. Current timestamp as a specific type\n```sql\n-- timestamp with time zone\nSELECT NOW()::timestamptz;\n\n-- timestamp without time zone\nSELECT NOW()::timestamp;\n```\n\n### 4. Using it in a query (e.g., inserting a row with the current time)\n```sql\nINSERT INTO my_table (name, created_at)\nVALUES ('Test', CURRENT_TIMESTAMP);\n`

### Check postgreSQL connection with python

In [9]:
import psycopg2
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Database configuration
DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
    "dbname": os.getenv("DB_NAME", "my_dbt_db"),
    "user": os.getenv("DB_USER", "dbt_user"),
    "password": os.getenv("DB_PASSWORD"),
}

sql = """SELECT * FROM transform.fact_appointments
        WHERE appointment_id = 'A0000625'"""

with psycopg2.connect(**DB_CONFIG) as connection:
    connection.set_session(readonly=True)

    with connection.cursor() as cursor:
        cursor.execute("SET LOCAL statement_timeout = '15000'")
        cursor.execute(sql)

        columns = [column[0] for column in cursor.description]
        rows = cursor.fetchmany(101)

        if len(rows) > 100:
            raise ValueError(
                "The query returned more than 100 rows. "
                "Please narrow your question."
            )

        print([dict(zip(columns, row))
                for row in rows])
        # print(rows)


[{'appointment_id': 'A0000625', 'patient_id': 'PT000376', 'provider_id': 'P011', 'appointment_date': datetime.date(2026, 4, 12), 'status': 'COMPLETED', 'updated_at': datetime.datetime(2026, 4, 12, 18, 0)}]


### Make LLM generate executable sql statement

In [10]:
# Database schema provided to the LLM
SCHEMA = """
Table: publish.pub_dim_patients
description: contains patient information

Columns:
- patient_id (VARCHAR(20), Not Null, Unique)
- first_name (VARCHAR(200)): Patient's first name.
- last_name (VARCHAR(200)): Patient's last name.
- date_of_birth (DATE): Patient's date of birth.
- phone (VARCHAR(30)): Patient's contact phone number.
- zip (VARCHAR(15)): ZIP or postal code associated with the patient.
- created_date (DATE): Date on which the patient record was created.

Table: publish.pub_dim_providers
description: Contains healthcare provider information

Columns:
- provider_id (VARCHAR(20), Not Null): Identifier assigned to the healthcare provider. Multiple records may exist for a provider when role or effective-date history is maintained.
- npi (VARCHAR(10)): National Provider Identifier assigned to the healthcare provider.
- first_name (VARCHAR(200)): Provider's first name.
- last_name (VARCHAR(200)): Provider's last name.
- specialty (VARCHAR(100)): Medical specialty of the provider.
- role (VARCHAR(50)): Provider's role within the healthcare organization.
- effective_date (DATE): Date from which the provider record becomes effective.
- location (VARCHAR(100)): Clinic or location associated with the provider.

Table: publish.pub_fact_referrals
description: Incremental table contains referral events

Columns:
- referral_id (VARCHAR(20), Not Null, Unique): Unique identifier assigned to each referral event.
- patient_id (VARCHAR(20), Not Null): Identifier of the patient associated with the referral.
- referring_provider_id (VARCHAR(20)): Identifier of the provider initiating the referral.
- receiving_provider_id (VARCHAR(20)): Identifier of the provider receiving the referral.
- referral_date (DATE): Date on which the referral was made.

Table: publish.pub_fact_appointments
description: Incremental table containing appointment data. New and updated records are merged using appointment_id.
Columns:
- appointment_id (VARCHAR(20), Not Null): Unique identifier assigned to an appointment.
- patient_id (VARCHAR(20)): Identifier of the patient associated with the appointment.
- provider_id (VARCHAR(20)): Identifier of the provider associated with the appointment.
- appointment_date (DATE): Date on which the appointment is scheduled.
- status (VARCHAR(30), Not Null, Allowed Values: 'CANCELLED', 'NO_SHOW', 'COMPLETED', 'SCHEDULED'): Current status of the appointment.
- updated_at (TIMESTAMP): Timestamp indicating when the appointment record was last updated.

Relationships:
- pub_fact_appointments.patient_id -> pub_dim_patients.patient_id, many-to-one relationship.
- pub_fact_appointments.provider_id -> pub_dim_providers.provider_id, many-to-many relationship
- pub_fact_appointments.patient_id -> pub_fact_referrals.patient_id, many-to-many relationship
- pub_fact_appointments.provider_id -> pub_fact_referrals.provider_id, many-to-many relationship
- pub_fact_referrals.patient_id -> pub_dim_patients.patient_id, many-to-one relationship
- pub_fact_referrals.referring_provider_id -> pub_dim_providers.provider_id, many-to-one relationship
- pub_fact_referrals.receiving_provider_id -> pub_dim_providers.provider_id, many-to-one relationship
- pub_fact_referrals.receiving_provider_id -> pub_fact_appointments.provider_id, many-to-one relationship

Use PostgreSQL SQL supported syntax.
"""

In [20]:
import re

def generate_sql(question: str) -> str:
    """
        Convert the user's question into a PostgreSQL SELECT query.
    Args:
        question: str
    Returns:
        query: str"""

    prompt = f"""
You are a PostgreSQL expert answering questions about clinic referrals
and appointments.

Database schema:
{SCHEMA}

Rules:
1. Generate one read-only SELECT query.
2. Use only the tables and columns listed in the schema.
3. Do not generate INSERT, UPDATE, DELETE, DROP, ALTER, or other
   data-modifying statements.
4. Use appropriate JOIN, GROUP BY, date filtering, and aggregations.
5. If user's question not relevent to the schema and tables, you should provide empty string.
6. Avoid counting a referral multiple times when several appointments
   match it. Use EXISTS or COUNT(DISTINCT referral_id) as appropriate.
7. Return only SQL, without Markdown or explanations.

User question:
{question}
"""
    messages=[
                {
                    "role": "system",
                    "content": "Generate accurate, read-only PostgreSQL query."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]
    response = llm_model2.invoke(messages)

    sql = response.content.strip()

    # Remove Markdown fences if the model returns them
    # sql = re.sub(r"^```(?:sql)?\s*", "", sql, flags=re.IGNORECASE)
    # sql = re.sub(r"\s*```$", "", sql)

    return sql.rstrip(";")

sql = generate_sql("Is gorilla still alive?")
print(sql)

In [17]:
def execute_sql(sql: str) -> list[dict]:
    """
    Execute the query in a read-only transaction.
    Args:
        sql: str -> llm generated query
    Returns:
        list[dict] -> list of resultant records"""

    with psycopg2.connect(**DB_CONFIG) as connection:
        connection.set_session(readonly=True)

        with connection.cursor() as cursor:
            cursor.execute("SET LOCAL statement_timeout = '15000'")
            cursor.execute(sql)

            columns = [column[0] for column in cursor.description]
            rows = cursor.fetchall()

            if len(rows) > 100:
                raise ValueError(
                    "The query returned more than 100 rows. "
                    "Please narrow your question."
                )

            return [
                dict(zip(columns, row))
                for row in rows
            ]

context =  execute_sql(sql)
print(context)

[{'status': 'COMPLETED'}]


In [21]:
import json

def generate_answer(question: str, sql: str, results: str):
    """
    Convert SQL results into a natural-language answer.
    Args:
        question: str -> user question
        sql: str -> executable sql query
        results: str -> query result
    Returns:
        answer: str -> llm's answer for the user question"""

    prompt = f"""
Answer the user's question using only the database results below.

User question:
{question}

SQL executed:
{sql}

Query results:
{json.dumps(results, default=str, indent=2)}

Instructions:
- Explain the results clearly and concisely.
- If no sql and query results are provided, then assume user's question do not has answer in our hospital db.
- If we have sql query but don't have query results, then assume we don't have relevant data for user's question.
- Do not invent facts that are not present in the results.
- Do not share the following patient related sensitive information - patient_id, name, first_name, last_name, date_of_birth, phone, zip.
"""

    messages=[
                {
                    "role": "system",
                    "content": "You provide appropriate answer to the user's question based only on provided prompt"
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]
    response = llm_model2.invoke(messages)

    return response.content.strip()

answer = generate_answer("Is gorilla still alive?", sql, context)
print(answer)


The database query completed, but the results do not contain any information about gorillas or their status. Therefore, I’m unable to determine from the available data whether a gorilla is still alive.
